# Fine-tune Cross-Encoder CV-JD v0.6

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` với **MSE regression loss** trên dataset v0.5 có class balance hoàn hảo.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss`, `BoundaryAwareLoss` (regression: label = score / 100) |
| **Evaluator** | Spearman correlation, LabelAcc |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Mục tiêu**: Vượt LabelAcc 60.76% của v0.2 nhờ dataset balance hoàn hảo (20% per class) + 91% more data (13,350 vs 7,000 pairs).

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.5/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Helper Functions

In [ ]:
import json
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from typing import Any

# Loss function: BoundaryAwareLoss
class BoundaryAwareLoss(torch.nn.Module):
    """MSE + ordinal BCE at boundaries (40/60/75/90)."""
    _BOUNDARIES = [0.40, 0.60, 0.75, 0.90]
    _ALPHA = 0.3

    def forward(self, preds: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        mse = F.mse_loss(preds, labels)
        ordinal = preds.new_zeros(1)
        for b in self._BOUNDARIES:
            target = (labels >= b).float()
            pred_logit = 20.0 * (preds - b)
            ordinal = ordinal + F.binary_cross_entropy_with_logits(pred_logit, target)
        return mse + self._ALPHA * ordinal / len(self._BOUNDARIES)

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

# Evaluator: CELabelAccEvaluator
class CELabelAccEvaluator:
    def __init__(self, sentence_pairs: list, labels_0_100: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_100 = labels_0_100
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CELabelAccEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_100=[ex.label * 100 for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        arr = np.asarray(preds)
        if arr.ndim == 2:
            pred_classes = arr.argmax(axis=1).tolist()
            true_classes = [_score_to_class_index(t) for t in self.labels_0_100]
            return sum(1 for p, t in zip(pred_classes, true_classes) if p == t) / len(pred_classes)
        pred_100 = [float(p) * 100 for p in preds]
        label_acc = sum(
            1 for p, t in zip(pred_100, self.labels_0_100)
            if abs(p - t) <= 10
        ) / len(pred_100)
        return label_acc

def _score_to_class_index(score: float) -> int:
    """Convert 0-100 score to 5-class index."""
    if score < 40:
        return 0
    elif score < 60:
        return 1
    elif score < 75:
        return 2
    elif score < 90:
        return 3
    else:
        return 4

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")
    
    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]
    
    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])
    
    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)
    
    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("Helper functions loaded.")

## Load Dataset

In [ ]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

## Run 1: MSELoss + Spearman, 10 epochs (Baseline)

In [ ]:
import torch

run_name = "v0.6-run1-mse-spearman"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")

print(f"🚀 Starting {run_name}...")
model.fit(
    train_objectives=[(train_examples, torch.nn.MSELoss())],
    evaluator=evaluator,
    epochs=10,
    batch_size=16,
    warmup_ratio=0.1,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

results_run1 = {
    'run': run_name,
    'loss': 'MSE',
    'evaluator': 'Spearman',
    'epochs': 10,
    'val': val_metrics,
    'test': test_metrics
}

## Run 2: BoundaryAwareLoss + Spearman, 10 epochs

In [ ]:
import torch

run_name = "v0.6-run2-boundary-spearman"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
boundary_loss = BoundaryAwareLoss()

print(f"🚀 Starting {run_name}...")
model.fit(
    train_objectives=[(train_examples, boundary_loss)],
    evaluator=evaluator,
    epochs=10,
    batch_size=16,
    warmup_ratio=0.1,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

results_run2 = {
    'run': run_name,
    'loss': 'BoundaryAwareLoss',
    'evaluator': 'Spearman',
    'epochs': 10,
    'val': val_metrics,
    'test': test_metrics
}

## Run 3: MSELoss + LabelAcc, 10 epochs

In [ ]:
import torch

run_name = "v0.6-run3-mse-labelacc"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

evaluator = CELabelAccEvaluator.from_input_examples(val_examples, name="val_label_acc")

print(f"🚀 Starting {run_name}...")
model.fit(
    train_objectives=[(train_examples, torch.nn.MSELoss())],
    evaluator=evaluator,
    epochs=10,
    batch_size=16,
    warmup_ratio=0.1,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

results_run3 = {
    'run': run_name,
    'loss': 'MSE',
    'evaluator': 'LabelAcc',
    'epochs': 10,
    'val': val_metrics,
    'test': test_metrics
}

## Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Run': 'Run 1 (MSE+Spearman)',
        'Val LabelAcc': results_run1['val']['LabelAcc'],
        'Test LabelAcc': results_run1['test']['LabelAcc'],
        'Test MAE': results_run1['test']['MAE'],
        'Test RMSE': results_run1['test']['RMSE']
    },
    {
        'Run': 'Run 2 (Boundary+Spearman)',
        'Val LabelAcc': results_run2['val']['LabelAcc'],
        'Test LabelAcc': results_run2['test']['LabelAcc'],
        'Test MAE': results_run2['test']['MAE'],
        'Test RMSE': results_run2['test']['RMSE']
    },
    {
        'Run': 'Run 3 (MSE+LabelAcc)',
        'Val LabelAcc': results_run3['val']['LabelAcc'],
        'Test LabelAcc': results_run3['test']['LabelAcc'],
        'Test MAE': results_run3['test']['MAE'],
        'Test RMSE': results_run3['test']['RMSE']
    }
])

print("\n📊 Experiment Summary (v0.6 vs v0.2 baseline: 60.76%)\n")
print(summary.to_string(index=False))

best_idx = summary['Test LabelAcc'].idxmax()
best_run = summary.loc[best_idx]
print(f"\n🏆 Best run: {best_run['Run']} with {best_run['Test LabelAcc']:.2%} test LabelAcc")
print(f"  Ceiling broken: {'✅ YES' if best_run['Test LabelAcc'] > 0.6076 else '❌ NO'}")
print(f"  Improvement over v0.2: {(best_run['Test LabelAcc'] - 0.6076) * 100:+.2f} pp")

## Save Reports

In [ ]:
import os
from pathlib import Path

# Create reports directory
os.makedirs('artifacts/reports', exist_ok=True)

# Save individual run reports
for run_results in [results_run1, results_run2, results_run3]:
    report = {
        'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
        'dataset_version': 'v0.5',
        'dataset_size': {'train': len(train_examples), 'val': len(val_examples), 'test': len(test_examples)},
        'run': run_results['run'],\n        'loss': run_results['loss'],
        'evaluator': run_results['evaluator'],
        'epochs': run_results['epochs'],
        'metrics': {
            'validation': run_results['val'],
            'test': run_results['test']
        }
    }
    report_path = f\"artifacts/reports/{run_results['run']}_report.json\"\n    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f\"✅ Saved {report_path}\")\n\n# Create consolidated report\nconsolidated = {
    'experiment': 'cross-encoder-v0.6',
    'dataset': 'v0.5',
    'dataset_size': {
        'train': len(train_examples),
        'val': len(val_examples),
        'test': len(test_examples),
        'total': len(train_examples) + len(val_examples) + len(test_examples)
    },
    'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'baseline_v0_2': {
        'dataset_size': 7000,
        'test_label_acc': 0.6076,
        'loss': 'MSE',
        'evaluator': 'Spearman',
        'epochs': 10
    },
    'runs': [\n        {\n            'name': 'Run 1: MSELoss + Spearman',
            'loss': results_run1['loss'],
            'evaluator': results_run1['evaluator'],
            'epochs': results_run1['epochs'],
            'val': results_run1['val'],
            'test': results_run1['test']\n        },
        {\n            'name': 'Run 2: BoundaryAwareLoss + Spearman',
            'loss': results_run2['loss'],
            'evaluator': results_run2['evaluator'],
            'epochs': results_run2['epochs'],
            'val': results_run2['val'],
            'test': results_run2['test']\n        },
        {\n            'name': 'Run 3: MSELoss + LabelAcc',
            'loss': results_run3['loss'],
            'evaluator': results_run3['evaluator'],
            'epochs': results_run3['epochs'],
            'val': results_run3['val'],
            'test': results_run3['test']\n        }\n    ]\n}\n\n# Add findings\nbest_test_labelacc = max(results_run1['test']['LabelAcc'], results_run2['test']['LabelAcc'], results_run3['test']['LabelAcc'])\nconsolidated['findings'] = {\n    'best_test_label_acc': best_test_labelacc,\n    'ceiling_broken': best_test_labelacc > 0.6076,
    'improvement_over_v0_2_pp': (best_test_labelacc - 0.6076) * 100\n}\n\nreport_path = 'artifacts/reports/v0.6_complete_report.json'\nwith open(report_path, 'w') as f:\n    json.dump(consolidated, f, indent=2)\n\nprint(f\"✅ Consolidated report saved to {report_path}\")

## Save to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Find best run by test LabelAcc
best_test_labelacc = max(
    results_run1['test']['LabelAcc'],
    results_run2['test']['LabelAcc'],
    results_run3['test']['LabelAcc']
)

if results_run1['test']['LabelAcc'] == best_test_labelacc:
    best_model_dir = 'artifacts/models/cross-encoder-cv-jd-v0.6-run1-mse-spearman'
    best_run_name = 'Run 1 (MSE + Spearman)'
elif results_run2['test']['LabelAcc'] == best_test_labelacc:
    best_model_dir = 'artifacts/models/cross-encoder-cv-jd-v0.6-run2-boundary-spearman'
    best_run_name = 'Run 2 (BoundaryAware + Spearman)'
else:
    best_model_dir = 'artifacts/models/cross-encoder-cv-jd-v0.6-run3-mse-labelacc'
    best_run_name = 'Run 3 (MSE + LabelAcc)'

# Copy only best model with new name
dest_model = f"{drive_base}/models/cross-encoder-cv-jd-v0.6"
if os.path.exists(dest_model):
    shutil.rmtree(dest_model)
shutil.copytree(best_model_dir, dest_model)
print(f"✅ Best model ({best_run_name}): {best_test_labelacc:.2%}")
print(f"✅ Saved to {dest_model}")

# Copy all reports
for report_file in os.listdir('artifacts/reports'):
    if report_file.endswith('.json'):
        src = f"artifacts/reports/{report_file}"
        dest = f"{drive_base}/reports/{report_file}"
        shutil.copy(src, dest)
        print(f"✅ Copied {report_file}")

print(f"\n✅ Done! Best model: cross-encoder-cv-jd-v0.6")